# Simulation Evaluation

Evaluate fitted models against ground truth: convergence diagnostics,
mutation effect accuracy, shift sparsity, replicate correlations,
variant phenotype predictions, and cross-validation.

In [ ]:
config_path = "config/config.yaml"
profile = None

In [ ]:
import warnings
import os
import pickle
import itertools
from collections import defaultdict

import pandas as pd
import numpy as np
from plotnine import *
import multidms

from notebooks._common import load_config, build_phenotype_fxn_dict, reconstruct_simulators

In [ ]:
cfg = load_config(config_path, profile)
sim = cfg["simulation"]
fit = sim["fitting"]

seed = cfg["seed"]
train_frac = cfg["train_frac"]
output_dir = sim["output_dir"]
gpu_ids = cfg["gpu_ids"]
n_processes = cfg["n_processes"]
lasso_choice = sim["lasso_choice"]
wt_latent = sim["wt_latent"]
sigmoid_phenotype_scale = sim["sigmoid_phenotype_scale"]

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
%matplotlib inline

## Load data

In [ ]:
fit_collection_df = pickle.load(open(f"{output_dir}/fit_collection.pkl", "rb"))
mut_effects_df = pd.read_csv(f"{output_dir}/simulated_muteffects.csv")
func_scores = pd.read_csv(f"{output_dir}/simulated_func_scores.csv")

func_scores["func_score_type"] = pd.Categorical(
    func_scores["func_score_type"],
    categories=["observed_phenotype", "loose_bottle", "tight_bottle"],
    ordered=True,
)

print(f"Loaded {len(fit_collection_df)} fitted models, {len(mut_effects_df)} mutations, {len(func_scores)} functional scores")

## Model convergence

In [ ]:
model_collection = multidms.model_collection.ModelCollection(fit_collection_df)
print("Convergence counts:")
print(model_collection.fit_models.converged.value_counts())

In [ ]:
data = model_collection.convergence_trajectory_df(
    id_vars=("fusionreg", "measurement_type", "library")
)
data.index.name = "step"
data.reset_index(inplace=True)

p = (
    ggplot(
        data.query(f"iteration >= 0 & library == 'lib_1' & fusionreg == {lasso_choice}"),
        aes(x="iteration", y="objective_error_trajectory", linetype="library"),
    )
    + geom_line()
    + facet_wrap("~fusionreg+measurement_type", scales="free_y", ncol=3)
    + theme(figure_size=(10, 4))
)
_ = p.draw(show=True)

## Model vs. truth mutational effects

In [ ]:
model_collection = multidms.model_collection.ModelCollection(fit_collection_df)

groupby = ("library", "measurement_type", "fusionreg")
collection_muts_df = (
    model_collection.split_apply_combine_muts(groupby=groupby)
    .reset_index()
    .rename({"beta_h1": "predicted_beta", "shift_h2": "predicted_shift_h2"}, axis=1)
    .merge(
        mut_effects_df.rename(
            {"beta_h1": "true_beta", "beta_h2": "true_beta_h2", "shift": "true_shift"}, axis=1
        ),
        on="mutation",
    )
)
print(f"collection_muts_df: {collection_muts_df.shape[0]} rows")

In [ ]:
def series_corr(y_true, y_pred):
    return np.corrcoef(y_true, y_pred)[0, 1]

def series_mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

new_fit_models_cols = defaultdict(list)
for group, model_mutations_df in collection_muts_df.groupby(list(groupby)):
    for i, attribute in enumerate(group):
        new_fit_models_cols[groupby[i]].append(group[i])
    for parameter in ["beta", "shift"]:
        for metric_fxn, name in zip([series_corr, series_mae], ["corr", "mae"]):
            postfix = "_h2" if parameter == "shift" else ""
            y_pred = model_mutations_df[f"predicted_{parameter}{postfix}"]
            y_true = model_mutations_df[f"true_{parameter}"]
            new_fit_models_cols[f"{parameter}_{name}"].append(metric_fxn(y_true, y_pred))

model_collection.fit_models = model_collection.fit_models.merge(
    pd.DataFrame(new_fit_models_cols), on=list(groupby)
)

In [ ]:
metric = "corr"
data = (
    model_collection.fit_models.assign(
        measurement_library=lambda x: x["measurement_type"].astype(str) + " " + x["library"]
    ).melt(
        id_vars=list(groupby) + ["measurement_library"],
        value_vars=[f"beta_{metric}", f"shift_{metric}"],
        var_name="parameter",
        value_name=metric,
    )
)
data["measurement_type"] = pd.Categorical(
    data["measurement_type"],
    categories=["observed_phenotype", "loose_bottle", "tight_bottle"],
    ordered=True,
)
data["parameter"] = data["parameter"].str.replace(f"_{metric}", "")
data["parameter"] = pd.Categorical(data["parameter"], categories=["shift", "beta"], ordered=True)

for parameter, parameter_df in data.groupby("parameter", observed=True):
    p = (
        ggplot(parameter_df)
        + geom_line(aes(x="fusionreg", y=metric, group="measurement_library"))
        + geom_point(aes(x="fusionreg", y=metric, shape="library"), size=4)
        + facet_wrap("measurement_type", scales="free_y")
        + theme_classic()
        + theme(figure_size=(6, 3.3), axis_text_x=element_text(angle=90))
        + labs(title=f"Prediction vs. Ground Truth: {parameter}", x="lasso penalty (λ)", y="Correlation")
    )
    _ = p.draw(show=True)

data.to_csv(f"{output_dir}/model_vs_truth_beta_shift.csv", index=False)
print(f"Saved model_vs_truth_beta_shift.csv ({len(data)} rows)")

## Shift sparsity

In [ ]:
from dms_variants.constants import CBPALETTE

chart, sparsity_data = model_collection.shift_sparsity(return_data=True)
sparsity_data = sparsity_data.assign(
    library=sparsity_data.dataset_name.str.split("_").str[:2].str.join("_"),
    library_type=sparsity_data.dataset_name.str.split("_").str[:2].str.join("_") + "-" + sparsity_data.mut_type,
    measurement_type=sparsity_data.dataset_name.str.split("_").str[2:4].str.join("_"),
)
sparsity_data["measurement_type"] = pd.Categorical(
    sparsity_data["measurement_type"],
    categories=["observed_phenotype", "loose_bottle", "tight_bottle"],
    ordered=True,
)
sparsity_data["fusionreg"] = sparsity_data.fusionreg.astype(object)

true_sparsity = 1 - mut_effects_df.shifted_site.mean()

p = (
    ggplot(sparsity_data, aes("fusionreg", "sparsity"))
    + geom_hline(yintercept=true_sparsity, linetype="dashed", color="red")
    + geom_line(aes(group="library_type"))
    + geom_point(aes(fill="mut_type", shape="library"), size=4)
    + scale_fill_manual(values=CBPALETTE)
    + facet_wrap("measurement_type")
    + theme_classic()
    + theme(figure_size=(6, 3.3), axis_text_x=element_text(angle=90))
    + labs(x="lasso penalty (λ)", y="sparsity")
)
_ = p.draw(show=True)

sparsity_data.to_csv(f"{output_dir}/fit_sparsity.csv", index=False)
print(f"Saved fit_sparsity.csv ({len(sparsity_data)} rows)")

## Replicate correlations

In [ ]:
chart, corr_data = model_collection.mut_param_dataset_correlation(return_data=True)

corr_data = (
    corr_data.assign(
        lib1=corr_data.datasets.str.split(",").str[0],
        lib2=corr_data.datasets.str.split(",").str[1],
        measurement_type_1=lambda x: x["lib1"].str.split("_").str[2:4].str.join("_"),
        measurement_type_2=lambda x: x["lib2"].str.split("_").str[2:4].str.join("_"),
    )
    .query("(measurement_type_1 == measurement_type_2) & (~mut_param.str.contains('predicted_func_score'))")
    .rename({"measurement_type_1": "measurement_type"}, axis=1)
    .replace({"shift_h2": "shift"})
    .drop(["lib1", "lib2", "datasets", "measurement_type_2"], axis=1)
)

corr_data["measurement_type"] = pd.Categorical(
    corr_data["measurement_type"],
    categories=["observed_phenotype", "loose_bottle", "tight_bottle"],
    ordered=True,
)
corr_data["mut_param"] = pd.Categorical(corr_data["mut_param"], categories=["shift", "beta"], ordered=True)
corr_data["fusionreg"] = corr_data.fusionreg.astype(object)

for parameter, parameter_df in corr_data.groupby("mut_param", observed=True):
    p = (
        ggplot(parameter_df)
        + geom_line(aes(x="fusionreg", y="correlation", group="measurement_type"))
        + geom_point(aes(x="fusionreg", y="correlation"), size=4)
        + facet_wrap("measurement_type", scales="free_y")
        + theme_classic()
        + theme(figure_size=(6, 3.3), axis_text_x=element_text(angle=90))
        + labs(title=f"Library Replicate Correlations: {parameter}", x="lasso penalty (λ)", y="pearsonr")
    )
    _ = p.draw(show=True)

corr_data.to_csv(f"{output_dir}/library_replicate_correlation.csv", index=False)
print(f"Saved library_replicate_correlation.csv ({len(corr_data)} rows)")

## Model vs. truth variant phenotypes

In [ ]:
phenotype_fxn_dict_h1, phenotype_fxn_dict_h2 = reconstruct_simulators(
    sim, mut_effects_df, seed
)
print("Phenotype functions reconstructed")

In [ ]:
variants_df = pd.concat([
    row.model.get_variants_df(phenotype_as_effect=False)
    .assign(
        library=row.library,
        measurement_type=row.measurement_type,
        fusionreg=row.fusionreg,
    )
    .rename({
        "predicted_func_score": "predicted_phenotype",
        "predicted_latent": "predicted_latent_phenotype",
        "func_score": "measured_phenotype",
    }, axis=1)
    .assign(
        predicted_enrichment=lambda x: 2 ** x["predicted_phenotype"],
        measured_enrichment=lambda x: 2 ** x["measured_phenotype"],
        fit_idx=idx,
    )
    for idx, row in fit_collection_df.iterrows()
])

# Add ground truth phenotypes
variants_df = pd.concat([
    variants_df.query("condition == @homolog").assign(
        true_latent_phenotype=lambda x: x["aa_substitutions"].apply(pfxn["latentPhenotype"]),
        true_observed_phenotype=lambda x: x["aa_substitutions"].apply(pfxn["observedPhenotype"]),
        true_enrichment=lambda x: x["aa_substitutions"].apply(pfxn["observedEnrichment"]),
    )
    for homolog, pfxn in [("h1", phenotype_fxn_dict_h1), ("h2", phenotype_fxn_dict_h2)]
])

variants_df["measurement_type"] = pd.Categorical(
    variants_df["measurement_type"],
    categories=["observed_phenotype", "loose_bottle", "tight_bottle"],
    ordered=True,
)

In [ ]:
# Add variant phenotype metrics to fit_collection_df
for idx, model_variants_df in variants_df.groupby("fit_idx"):
    for metric_fxn, metric_name in zip([series_corr, series_mae], ["corr", "mae"]):
        fit_collection_df.loc[idx, f"variant_phenotype_{metric_name}"] = metric_fxn(
            model_variants_df["measured_phenotype"],
            model_variants_df["predicted_phenotype"],
        )

pheno_data = fit_collection_df[
    ["library", "measurement_type", "fusionreg", "dataset_name",
     "variant_phenotype_corr", "variant_phenotype_mae"]
]

p = (
    ggplot(
        pheno_data.assign(
            measurement_library=lambda x: x["library"] + "_" + x["measurement_type"].astype(str)
        )
    )
    + geom_line(aes(x="fusionreg", y="variant_phenotype_corr", group="measurement_library"))
    + geom_point(aes(x="fusionreg", y="variant_phenotype_corr", shape="library"), size=4)
    + facet_wrap("measurement_type", scales="free_y")
    + theme_classic()
    + theme(figure_size=(6.5, 3.3), axis_text_x=element_text(angle=90))
    + labs(title="Predicted vs. True Variant Phenotype", x="lasso penalty (λ)", y="pearsonr")
)
_ = p.draw(show=True)

pheno_data.to_csv(f"{output_dir}/model_vs_truth_variant_phenotype.csv", index=False)
print(f"Saved model_vs_truth_variant_phenotype.csv ({len(pheno_data)} rows)")

## Cross-validation

In [ ]:
train, test = [], {}
for (library, measurement), fs_df in func_scores.rename(columns={"homolog": "condition"}).groupby(
    ["library", "func_score_type"]
):
    if "enrichment" in measurement:
        continue

    wt_mask = fs_df["aa_substitutions"].fillna("").str.strip() == ""
    wt_rows = fs_df[wt_mask]
    non_wt_rows = fs_df[~wt_mask].sample(frac=1, random_state=seed)

    n_train = int(len(non_wt_rows) * train_frac)
    train_split = pd.concat([wt_rows, non_wt_rows.iloc[:n_train]])
    test_split = non_wt_rows.iloc[n_train:]

    name = f"{library}_{measurement}"

    train_split = train_split.copy()
    train_split["aa_substitutions"] = train_split["aa_substitutions"].fillna("")

    train.append(
        multidms.Data(
            train_split,
            reference="h1",
            alphabet=multidms.AAS_WITHSTOP_WITHGAP,
            verbose=False,
            name=name,
        )
    )
    test[name] = test_split

print(f"Created {len(train)} train datasets, {len(test)} test datasets")

In [ ]:
cv_fitting_params = {
    "maxiter": [fit["maxiter"]],
    "tol": [fit["tol"]],
    "fusionreg": fit["fusionreg_values"],
    "l2reg": [fit["l2reg"]],
    "beta0_ridge": [fit["beta0_ridge"]],
    "ge_type": [fit["ge_type"]],
    "ge_kwargs": [fit["ge_kwargs"]],
    "cal_kwargs": [fit["cal_kwargs"]],
    "loss_kwargs": [fit["loss_kwargs"]],
    "warmstart": [fit["warmstart"]],
    "beta0_init": [fit["beta0_init"]],
    "alpha_init": [fit["alpha_init"]],
    "beta_clip_range": [tuple(fit["beta_clip_range"])],
    "dataset": train,
}

_, _, fit_collection_cv = multidms.model_collection.fit_models(
    cv_fitting_params, gpu_ids=gpu_ids, n_processes=n_processes
)

for col in fit_collection_cv.columns:
    if fit_collection_cv[col].apply(lambda x: isinstance(x, dict)).any():
        fit_collection_cv[col] = fit_collection_cv[col].apply(str)

In [ ]:
mc = multidms.model_collection.ModelCollection(fit_collection_cv)

# Filter test data to only include variants whose mutations were all seen during training
filtered_test = {}
for name, test_df in test.items():
    model = mc.fit_models.query(f"dataset_name == '{name}'").iloc[0].model
    known_muts = set(model.data.mutations)

    def all_muts_known(subs):
        if pd.isna(subs) or subs.strip() == "":
            return True
        return all(m in known_muts for m in subs.split())

    mask = test_df["aa_substitutions"].apply(all_muts_known)
    filtered_test[name] = test_df[mask]
    n_dropped = (~mask).sum()
    if n_dropped > 0:
        print(f"{name}: dropped {n_dropped}/{len(test_df)} test variants with unseen mutations")

mc.add_eval_loss(filtered_test, overwrite=True)

In [ ]:
cv_data = mc.fit_models.copy()
cv_data["library"] = cv_data["dataset_name"].str.split("_").str[0:2].str.join("_")
cv_data["measurement_type"] = cv_data["dataset_name"].str.split("_").str[2:4].str.join("_")
cv_data["measurement_type"] = pd.Categorical(
    cv_data["measurement_type"],
    categories=["observed_phenotype", "loose_bottle", "tight_bottle"],
    ordered=True,
)

cv_data = (
    cv_data.melt(
        id_vars=["fusionreg", "library", "measurement_type"],
        value_vars=["total_loss_training", "total_loss_validation"],
        var_name="dataset",
        value_name="loss",
    )
    .assign(
        dataset=lambda x: x["dataset"].str.replace("total_loss_", ""),
        lib_dataset=lambda x: x["library"] + " " + x["dataset"],
    )
)

p = (
    ggplot(cv_data)
    + geom_line(aes(x="fusionreg", y="loss", group="lib_dataset"))
    + geom_point(aes(x="fusionreg", y="loss", fill="dataset", shape="library"), size=4)
    + facet_wrap("measurement_type", scales="free_y")
    + theme_classic()
    + theme(figure_size=(6.5, 3.3), axis_text_x=element_text(angle=90))
    + labs(x="lasso penalty (λ)", y="Huber loss w/o penalty")
)
_ = p.draw(show=True)

cv_data.to_csv(f"{output_dir}/cross_validation_loss.csv", index=False)
print(f"Saved cross_validation_loss.csv ({len(cv_data)} rows)")